# Plotting the YAOKI telemetry and telecommands (TMTC)

This notebook reads the parquet files containing telemetry and telecommands and plots out several interesting data streams.
Files are organized under the directory yaoki_parquet:
- manifest.yaml describes the tree of data.
- commands are sent to the rover
- status bits describe rover state
- telemetry are data packets sent from the rover to mission control


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import yaml
import os
import re

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_colwidth", None)

def plotly_timestamp(value):
    """Return a timezone-aware Python datetime rounded to Plotly-safe microseconds."""
    return pd.Timestamp(value).round("us").to_pydatetime()

def plotly_timestamps(values):
    """Return timezone-aware Python datetimes rounded to Plotly-safe microseconds."""
    timestamps = pd.to_datetime(values, utc=True)
    if isinstance(timestamps, pd.Series):
        timestamps = pd.DatetimeIndex(timestamps)
    return timestamps.round("us").to_pydatetime()



In [ ]:
PARQUET_DIR = Path("yaoki_parquet")
MANIFEST = yaml.safe_load((PARQUET_DIR / "manifest.yaml").read_text())
TELEMETRY = MANIFEST["telemetry_streams"]

telemetry_metadata = {
    name: yaml.safe_load((PARQUET_DIR / entry["metadata"]).read_text())
    for name, entry in TELEMETRY.items()
}
command_metadata = yaml.safe_load((PARQUET_DIR / MANIFEST["commands"]["metadata"]).read_text())

telemetry_labels = {
    name: metadata.get("display_name", name.rsplit("/", 1)[-1])
    for name, metadata in telemetry_metadata.items()
}

In [ ]:
# Mission Timeline Constants
IM2_LAUNCH_DATE = "2025-02-27T00:16:30Z"
IM2_LANDING_DATE = "2025-03-06T17:28:50Z"
IM2_FIRST_TELEMETRY_DATE = "2025-02-26T18:50:58.457Z"
IM2_LAST_TELEMETRY_DATE = "2025-03-07T05:42:56.476Z"

YAOKI_FIRST_TELEMETRY_DATE = "2025-03-07 02:17:15.150000+00:00"
YAOKI_LAST_TELEMETRY_DATE = "2025-03-07 04:33:04.389000+00:00"

IMAGE_TOTAL_PACKETS = 960
WIRE_CUT_DURATION_SECONDS = 200
NO_RX_TIMEOUT_MINUTES = 3
TEMPERATURE_LIMIT_CELSIUS = -40

start = datetime.fromisoformat(YAOKI_FIRST_TELEMETRY_DATE)
stop = datetime.fromisoformat(YAOKI_LAST_TELEMETRY_DATE)
mission_duration = stop - start

print(stop)
print(mission_duration)


## Direct Parquet Examples


In [ ]:
df_temperature0 = pd.read_parquet(PARQUET_DIR / TELEMETRY["/YAOKI/Lander/temperature0"]["file"])
df_temperature0.head(10)


In [ ]:
df_last_packet = pd.read_parquet(PARQUET_DIR / TELEMETRY["/YAOKI/Rover/last_packet_num"]["file"])
df_last_packet.sort_values("generation_time", ascending=False).head(10)


## Analyze Temperature Data


In [ ]:
temperature_paths = [
    *(f"/YAOKI/Rover/ADC_TEMP{i}" for i in range(8)),
    *(f"/YAOKI/Lander/temperature{i}" for i in range(2)),
]

frames = []
for path in temperature_paths:
    df_param = pd.read_parquet(PARQUET_DIR / TELEMETRY[path]["file"])
    if df_param.empty:
        continue
    df_param = df_param.rename(columns={"generation_time": "timestamp"})
    df_param["label"] = telemetry_labels[path]
    df_param["value"] = df_param["eng_value"].where(df_param["eng_value"].notna(), df_param["raw_value"])
    frames.append(df_param[["label", "timestamp", "value"]])

df = pd.concat(frames, ignore_index=True)
df.head()


In [ ]:
df[df['timestamp'] < '2/28/2025']

In [ ]:
fig = px.scatter(
    df,
    x="timestamp",
    y="value",
    color="label",
    title="Yaoki Surface OPS: Temperature Sensors",
    labels={"value": "Temperature (deg C)", "timestamp": "Time (UTC)"},
)
fig.add_hline(
    y=TEMPERATURE_LIMIT_CELSIUS,
    line_color="black",
    annotation_text="rover temperature sensor limit",
    annotation_position="bottom",
)
fig.add_vline(x=start.astimezone(timezone.utc), line_dash="dash", line_color="black")
fig.add_vline(x=stop.astimezone(timezone.utc), line_dash="dash", line_color="black")
fig.update_layout(yaxis_range=[-65, 0])
fig.update_layout(xaxis_range=[start,stop])
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.4))

fig.show()


## Analyze Radio Data


In [ ]:
radio_paths = [
    "/YAOKI/Rover/last_rssi",
    "/YAOKI/Rover/Rx_cnt",
    "/YAOKI/Rover/RxError_cnt",
    "/YAOKI/Rover/Tx_cnt",
]

frames = []
for path in radio_paths:
    df_param = pd.read_parquet(PARQUET_DIR / TELEMETRY[path]["file"])
    if df_param.empty:
        continue
    df_param = df_param.rename(columns={"generation_time": "timestamp"})
    df_param["label"] = telemetry_labels[path]
    df_param["value"] = df_param["eng_value"].where(df_param["eng_value"].notna(), df_param["raw_value"])
    frames.append(df_param[["label", "timestamp", "value"]])

df = pd.concat(frames, ignore_index=True)
df.head()


In [ ]:
fig = px.scatter(
    df,
    x="timestamp",
    y="value",
    color="label",
    title="Yaoki Surface OPS: Radio RSSI and Telemetry Counters",
    labels={"value": "Value", "timestamp": "Time (UTC)"},
    template="simple_white",
)
fig.add_vline(x=start.astimezone(timezone.utc), line_dash="dash", line_color="black", line_width=3)
fig.add_vline(x=stop.astimezone(timezone.utc), line_dash="dash", line_color="black", line_width=3)
fig.update_layout(yaxis_range=[-100, 110])
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.3))
fig.show()


In [ ]:
max_n = 2
filterdf = df[df["label"] == telemetry_labels["/YAOKI/Rover/last_rssi"]]
filterdf.nlargest(max_n, "value").iloc[0:max_n]


## Analyze Camera Data


In [ ]:
camera_counter_paths = [
    "/YAOKI/Rover/CamSend_num",
    "/YAOKI/Rover/CamSent_cnt",
]

frames = []
for path in camera_counter_paths:
    df_param = pd.read_parquet(PARQUET_DIR / TELEMETRY[path]["file"])
    if df_param.empty:
        continue
    df_param = df_param.rename(columns={"generation_time": "timestamp"})
    df_param["label"] = telemetry_labels[path]
    df_param["value"] = df_param["eng_value"].where(df_param["eng_value"].notna(), df_param["raw_value"])
    frames.append(df_param[["label", "timestamp", "value"]])

df = pd.concat(frames, ignore_index=True)
df.head()


In [ ]:
fig = px.scatter(
    df,
    x="timestamp",
    y="value",
    color="label",
    title="Yaoki Surface OPS: Image Counters",
    labels={"value": "Value", "timestamp": "Time (UTC)"},
    template="simple_white",
)
fig.add_vline(x=start.astimezone(timezone.utc), line_dash="dash", line_color="black", line_width=3)
fig.add_vline(x=stop.astimezone(timezone.utc), line_dash="dash", line_color="black", line_width=3)
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.2))
fig.show()


In [ ]:
packet_paths = [
    "/YAOKI/Rover/last_packet_num",
    "/YAOKI/Rover/missing_packet_cnt",
]

frames = []
for path in packet_paths:
    df_param = pd.read_parquet(PARQUET_DIR / TELEMETRY[path]["file"])
    if df_param.empty:
        continue
    df_param = df_param.rename(columns={"generation_time": "timestamp"})
    df_param["label"] = telemetry_labels[path]
    df_param["value"] = df_param["eng_value"].where(df_param["eng_value"].notna(), df_param["raw_value"])
    frames.append(df_param[["label", "timestamp", "value"]])

df = pd.concat(frames, ignore_index=True)
df.head()


In [ ]:
df_filtered = df[df["label"] == telemetry_labels["/YAOKI/Rover/missing_packet_cnt"]]
df_filtered = df_filtered.sort_values("timestamp").reset_index(drop=True)

reset_mask = df_filtered["value"].diff() > 0
prev_vals = df_filtered["value"].shift(1)
last_before_reset = prev_vals[reset_mask].astype(int)
last_before_ts = df_filtered["timestamp"].shift(1)[reset_mask]

result = pd.DataFrame({
    "timestamp_last_frame": last_before_ts.values,
    "missing frames": last_before_reset.values,
})
result.index = result.index + 2
result["timestamp_last_frame"] = pd.to_datetime(result["timestamp_last_frame"], errors="coerce", utc=True)

result.at[9, "missing frames"] = result.loc[[9, 10], "missing frames"].sum()
result = result.drop(index=10)
result.reset_index(drop=True, inplace=True)
result.index = result.index + 2

print("statistics for nominal images:")
print(result["missing frames"].describe())
print("")
print(f"median of missing packets: {result['missing frames'].median() / IMAGE_TOTAL_PACKETS * 100:.2f}%")
print(f"mean of missing packets: {result['missing frames'].mean() / IMAGE_TOTAL_PACKETS * 100:.2f}%")
print(f"75% of images have less than: {result['missing frames'].quantile(.75) / IMAGE_TOTAL_PACKETS * 100:.2f}% missing packets")

result.loc[1, "missing frames"] = 18
result.loc[1, "timestamp_last_frame"] = pd.NaT


def insert_by_label(df, labels, new_row):
    df = df.copy()
    for lbl in sorted(labels):
        df.index = pd.Index([i + 1 if i >= lbl else i for i in df.index])
        df.loc[lbl] = new_row
    return df.sort_index()

result = insert_by_label(
    result,
    labels=[13, 25],
    new_row={"missing frames": IMAGE_TOTAL_PACKETS, "timestamp_last_frame": pd.NaT},
)

last_val = df_filtered["value"].iloc[-1]
last_ts = df_filtered["timestamp"].iloc[-1]
result.loc[result.index.max() + 1, "missing frames"] = last_val
result.loc[result.index.max(), "timestamp_last_frame"] = last_ts

result = result.sort_index()
result["timestamp_last_frame"] = pd.to_datetime(result["timestamp_last_frame"], errors="coerce", utc=True)
result


In [ ]:
fig = px.scatter(
    df,
    x="timestamp",
    y="value",
    color="label",
    labels={"value": "Value", "timestamp": "Time (UTC)"},
    template="simple_white",
)
for ts in result["timestamp_last_frame"].dropna():
    fig.add_vline(x=ts, line_dash="solid", line_color="orange", line_width=3)
fig.add_vline(x=start.astimezone(timezone.utc), line_dash="dash", line_color="black", line_width=3)
fig.add_vline(x=stop.astimezone(timezone.utc), line_dash="dash", line_color="black", line_width=3)
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.2))
fig.show()


In [ ]:
fig = px.bar(
    result,
    y="missing frames",
    text="missing frames",
    title="Yaoki Surface OPS: Missing Image Frames",
    labels={"missing frames": "Missing Frames (log scale)", "pre_reset_ts": "Time (UTC)"},
    template="simple_white",
)

colorway = pio.templates["simple_white"].layout.colorway
colors = [
    colorway[2] if idx == 1 else colorway[3] if idx in {13, 25} else colorway[1] if idx == 30 else colorway[0]
    for idx in result.index
]
fig.update_traces(marker_color=colors)
fig.update_xaxes(title_text="Capture Image ID", tickmode="linear", tick0=0, dtick=1, range=[0.5, result.index.max() + 0.5])
fig.update_yaxes(type="log", tickformat=".0f", nticks=10)
fig.update_traces(texttemplate="%{text}", textposition="outside", cliponaxis=False)
fig.show()


## Accelerometer Data Analysis


In [ ]:
TELEMETRY

In [ ]:
accelerometer_paths = [
    "YAOKI/Xr", # psuedonym for /YAOKI/Rover/Accelerometer_X that resolves correctly through yamcs bug that tolowers it.
    "/YAOKI/Rover/accelerometer_x",
    "YAOKI/Yr",
    "/YAOKI/Rover/accelerometer_y",
    "YAOKI/Zr",
    "/YAOKI/Rover/accelerometer_z"
]

frames = []
for path in accelerometer_paths:
    df_param = pd.read_parquet(PARQUET_DIR / TELEMETRY[path]["file"])
    if df_param.empty:
        continue
    df_param = df_param.rename(columns={"generation_time": "timestamp"})
    df_param["label"] = telemetry_labels[path]
    df_param["value"] = df_param["eng_value"].where(df_param["eng_value"].notna(), df_param["raw_value"])
    frames.append(df_param[["label", "timestamp", "value"]])

df = pd.concat(frames, ignore_index=True)
df.head()


In [ ]:
fig = px.scatter(
    df,
    x="timestamp",
    y="value",
    color="label",
    title="Yaoki Surface OPS: Accelerometer",
    labels={"value": "Value", "timestamp": "Time (UTC)"},
    template="simple_white",
)
fig.add_vline(x=start.astimezone(timezone.utc), line_dash="dash", line_color="black", line_width=3)
fig.add_vline(x=stop.astimezone(timezone.utc), line_dash="dash", line_color="black", line_width=3)
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.3))
fig.show()


## Status Bits Analysis

Eight status bits tell about the rover's state.  Telemetry samples arrive roughly every 30 s and we merge them into intervals via run-length encoding approach (telemetry_to_intervals function).

In [ ]:
df_status_tlm_sample = pd.read_parquet(PARQUET_DIR / TELEMETRY["/YAOKI/Rover/Status_CAM"]["file"])
df_status_tlm_sample

In [ ]:
_mission_end = pd.to_datetime(YAOKI_LAST_TELEMETRY_DATE, utc=True)

def telemetry_to_intervals(df, time_col="generation_time", value_col="eng_value", end=_mission_end):
    df = df.sort_values(time_col).reset_index(drop=True)
    df["group"] = (df[value_col] != df[value_col].shift()).cumsum()
    result = df.groupby("group").agg(
        status=(value_col, "first"),
        start=(time_col, "first"),
    ).reset_index(drop=True)
    result["stop"] = result["start"].shift(-1)
    result.loc[result.index[-1], "stop"] = end
    return result

status_paths = [
    "/YAOKI/Rover/Status_BAT",
    "/YAOKI/Rover/VBAT_CHECK",
    "/YAOKI/Rover/Status_ICUT",
    "/YAOKI/Rover/Status_WCUT",
    "/YAOKI/Rover/Status_CAM",
    "/YAOKI/Rover/Status_MOVE",
    "/YAOKI/Rover/Status_NORX",
    "/YAOKI/Rover/Status_TLM",
]

frames_tlm = []
status_legend_rows = []
for path in status_paths:
    df_tlm = pd.read_parquet(PARQUET_DIR / TELEMETRY[path]["file"])
    if df_tlm.empty:
        continue
    short_name = path.rsplit("/", 1)[-1]
    description = telemetry_labels[path]
    df_intervals = telemetry_to_intervals(df_tlm)
    df_intervals["status_name"] = short_name
    df_intervals["state"] = df_intervals["status"].map({0: "Off/0", 1: "On/1"})
    frames_tlm.append(df_intervals)
    status_legend_rows.append({"Short Name": short_name, "Description": description})

df_all_tlm = pd.concat(frames_tlm, ignore_index=True)
df_status_legend = pd.DataFrame(status_legend_rows)
df_all_tlm.head()

In [ ]:
fig = px.timeline(
    df_all_tlm,
    x_start="start",
    x_end="stop",
    y="status_name",
    color="state",
    title="YAOKI Rover Status Timeline (from Telemetry)",
    labels={"status_name": "Status", "state": "State"},
    template="simple_white",
)
fig.update_yaxes(title="", autorange="reversed")
fig.update_xaxes(title="Time (UTC)")
fig.show()

In [ ]:
df_status_legend.style.hide(axis="index")

In [ ]:
df_icut_tlm = pd.read_parquet(PARQUET_DIR / TELEMETRY["/YAOKI/Rover/Status_ICUT"]["file"])
df_icut_intervals = telemetry_to_intervals(df_icut_tlm)
on_intervals_tlm = df_icut_intervals[df_icut_intervals["status"] == 1]
total_on_time_tlm = (on_intervals_tlm["stop"] - on_intervals_tlm["start"]).sum()
total_on_time_tlm

## Telecommand Analysis


In [ ]:
df_cmds = pd.read_parquet(PARQUET_DIR / MANIFEST["commands"]["file"])
df_cmds


In [ ]:
telecmd_counts = df_cmds["Command Name"].value_counts().reset_index()
telecmd_counts.columns = ["Command Name", "count"]
telecmd_counts.loc[len(telecmd_counts)] = ["Total", telecmd_counts["count"].sum()]
telecmd_counts


In [ ]:
fig = px.pie(
    df_cmds,
    names="Command Name",
    title="Telecommand Distribution",
    template="simple_white",
    width=800,
    height=500,
)
fig.update_traces(textinfo="percent", texttemplate="%{percent:.0%}", insidetextorientation="radial")
fig.show()


In [ ]:
df_cmds[df_cmds["Command Name"] == "/YAOKI/Rover/CaptureCamera"]


In [ ]:
def compute_duration(row):
    cmd = row["Command Name"]
    args = row["Arguments"] or ""
    if cmd == "/YAOKI/Rover/CutWire":
        return pd.Timedelta(seconds=WIRE_CUT_DURATION_SECONDS)
    if cmd == "/YAOKI/Rover/RotateLeftAndRightMotors":
        nums = re.findall(r"-?\d+", args)
        return pd.Timedelta(seconds=max(abs(int(n)) for n in nums)) if nums else pd.Timedelta(seconds=60)
    return pd.Timedelta(minutes=1)


def extract_text(row):
    cmd = row["Command Name"]
    args = row["Arguments"] or ""
    if cmd == "/YAOKI/Rover/CutWire":
        m = re.search(r"-?\d+", args)
        return m.group() if m else ""
    if cmd == "/YAOKI/Rover/RotateLeftAndRightMotors":
        nums = re.findall(r"-?\d+", args)
        return str(max(abs(int(n)) for n in nums)) if nums else ""
    if cmd == "/YAOKI/Rover/CaptureCamera":
        return "SnF" if args == "cam: store_and_forward" else ""
    return ""

df_timeline = df_cmds.copy()
df_timeline["start"] = pd.to_datetime(df_timeline["Generation Time"], utc=True)
df_timeline["duration"] = df_timeline.apply(compute_duration, axis=1)
df_timeline["end"] = df_timeline["start"] + df_timeline["duration"]
df_timeline["text"] = df_timeline.apply(extract_text, axis=1)
df_timeline["cmd"] = df_timeline["Command Name"].str.replace(r"^/YAOKI/Rover/", "", regex=True)

fig = px.timeline(
    df_timeline,
    x_start="start",
    x_end="end",
    y="cmd",
    color="cmd",
    title="Yaoki Surface OPS: Telecommand Execution Timeline",
    labels={"start": "Start Time (UTC)", "end": "End Time (UTC)", "cmd": "Command"},
    template="simple_white",
)
for _, row in df_timeline.iterrows():
    if row["text"]:
        fig.add_annotation(
            x=row["end"],
            y=row["cmd"],
            text=row["text"],
            showarrow=False,
            xanchor="right",
            yanchor="top",
            font=dict(size=10, color="black"),
            yshift=17,
        )
fig.update_xaxes(title="Time (UTC)", range=[start, stop])
fig.update_layout(margin=dict(t=80, b=50, l=50, r=20))
fig


In [ ]:
fig = px.timeline(
    df_all_tlm,
    x_start="start",
    x_end="stop",
    y="status_name",
    color="state",
    title="YAOKI Rover Status + Telecommands",
    labels={"status_name": "Status", "state": "State"},
    template="simple_white",
)
fig.update_traces(opacity=0.2, selector=dict(type="bar"))
fig.update_yaxes(title="", autorange="reversed")
fig.update_xaxes(title="Time (UTC)")
fig.update_layout(
    yaxis2=dict(anchor="x", overlaying="y", side="right", range=[0, 1], showticklabels=False, showgrid=False),
    margin=dict(t=80, b=40, l=50, r=20),
)

colorway = pio.templates["simple_white"].layout.colorway
cmds = df_cmds["Command Name"].unique()
color_map = {cmd: colorway[i % len(colorway)] for i, cmd in enumerate(cmds)}

for cmd in cmds:
    group = df_cmds[df_cmds["Command Name"] == cmd]
    x_vals, y_vals = [], []
    for t in group["Generation Time"]:
        x_vals += [t, t, None]
        y_vals += [0, 1, None]
    fig.add_trace(go.Scatter(x=x_vals, y=y_vals, mode="lines", line=dict(color=color_map[cmd], width=2), name=cmd, yaxis="y2"))

fig.update_layout(legend=dict(orientation="h", y=-0.3, x=1, xanchor="right"))
fig.update_xaxes(range=[start, stop])
fig

## Lander Telemetry Analysis


In [ ]:
lander_paths = [
    "/YAOKI/Lander/temperature0",
    "/YAOKI/Lander/temperature1",
]

frames = []
for path in lander_paths:
    df_param = pd.read_parquet(PARQUET_DIR / TELEMETRY[path]["file"])
    if df_param.empty:
        continue
    df_param = df_param.rename(columns={"generation_time": "timestamp"})
    df_param["label"] = telemetry_labels[path]
    df_param["value"] = df_param["eng_value"].where(df_param["eng_value"].notna(), df_param["raw_value"])
    frames.append(df_param[["label", "timestamp", "value"]])

df = pd.concat(frames, ignore_index=True)
df.head()


In [ ]:
fig = px.scatter(df, x="timestamp", y="value", color="label", title="Temperature Telemetry over Time", template="simple_white")
fig.update_layout(
    legend=dict(
        orientation="h",        # horizontal
        yanchor="top",
        y=1.1,                 # below the x-axis
        xanchor="left",
        x=0
    )
)
fig.show()


In [ ]:
df_sorted = df.sort_values(["label", "timestamp"])
df_clean = df_sorted[df_sorted["value"].ne(df_sorted.groupby("label")["value"].shift())].reset_index(drop=True)
df_clean.head()


In [ ]:
fig = px.scatter(
    df_clean,
    x="timestamp",
    y="value",
    color="label",
    title="Temperature Telemetry over Time (cleaned)",
    template="simple_white",
)
fig.update_layout(
    legend=dict(
        orientation="h",        # horizontal
        yanchor="top",
        y=1.1,                 # below the x-axis
        xanchor="left",
        x=0.0
    )
)
yaoki_surface_start = plotly_timestamp(YAOKI_FIRST_TELEMETRY_DATE)
yaoki_surface_stop = plotly_timestamp(YAOKI_LAST_TELEMETRY_DATE)
yaoki_surface_midpoint = yaoki_surface_start + (yaoki_surface_stop - yaoki_surface_start) / 2
fig.add_vrect(x0=yaoki_surface_start, x1=yaoki_surface_stop, fillcolor="LightGray", opacity=0.3, layer="below", line_width=0)
fig.add_annotation(x=yaoki_surface_midpoint, yref="paper", y=0.25, text="YAOKI surface OPS", showarrow=False, font=dict(color="black", size=12), bgcolor="rgba(255,255,255,0.7)")
fig.add_vline(x=plotly_timestamp(IM2_LAUNCH_DATE), line_width=2)
fig.add_annotation(x=plotly_timestamp(IM2_LAUNCH_DATE), yref="paper", y=0.01, text="IM2 Launch", showarrow=False)
fig.add_vline(x=plotly_timestamp(IM2_LANDING_DATE), line_width=2)
fig.add_annotation(x=plotly_timestamp(IM2_LANDING_DATE), yref="paper", y=0.01, text="IM2 Landing", showarrow=False)
fig.update_layout(xaxis_title="Remote Time (UTC)", yaxis_title="Temperature (deg C)")
fig.show()


## Interpolation and Extrapolation (Experimental)


In [ ]:
IM2_THERMAL_SYSTEM_CHECK = "2025-03-06 23:32:51.009000+00:00"
im2_landing = pd.to_datetime(IM2_LANDING_DATE, utc=True)
im2_thermal_system_check = pd.to_datetime(IM2_THERMAL_SYSTEM_CHECK, utc=True)
yaoki_last_telemetry = pd.to_datetime(YAOKI_LAST_TELEMETRY_DATE, utc=True)

df_filtered = df_clean[df_clean["timestamp"] >= im2_landing].copy()
fig = go.Figure()
poly_degree = 3
end_seconds = (yaoki_last_telemetry - im2_landing).total_seconds()
check_seconds = (im2_thermal_system_check - im2_landing).total_seconds()

colorway = pio.templates["simple_white"].layout.colorway
sensor_list = [
    telemetry_labels["/YAOKI/Lander/temperature0"],
    telemetry_labels["/YAOKI/Lander/temperature1"],
]
sensor_color_map = {sensor: colorway[i % len(colorway)] for i, sensor in enumerate(sensor_list)}

for sensor in sensor_list:
    df_sensor = df_filtered[df_filtered["label"] == sensor].copy()
    if df_sensor.empty:
        continue
    df_sensor["seconds_since_landing"] = (df_sensor["timestamp"] - im2_landing).dt.total_seconds()
    x_all = df_sensor["seconds_since_landing"].values
    y_all = df_sensor["value"].values

    coeffs_poly = np.polyfit(x_all, y_all, deg=poly_degree)
    poly_fn = np.poly1d(coeffs_poly)
    x_poly_fit = np.linspace(x_all.min(), x_all.max(), 100)
    y_poly_fit = poly_fn(x_poly_fit)
    timestamps_poly = plotly_timestamps([im2_landing + pd.Timedelta(seconds=float(sec)) for sec in x_poly_fit])

    df_check = df_sensor[df_sensor["seconds_since_landing"] >= check_seconds]
    color = sensor_color_map[sensor]

    fig.add_trace(go.Scatter(x=plotly_timestamps(df_sensor["timestamp"]), y=df_sensor["value"], mode="markers", name=f"{sensor} Data", marker=dict(color=color, symbol="circle", size=6)))
    fig.add_trace(go.Scatter(x=timestamps_poly, y=y_poly_fit, mode="lines", line=dict(color=color, dash="dash", width=2), name=f"{sensor} Poly Fit (deg={poly_degree})"))

    if not df_check.empty:
        coeffs_lin = np.polyfit(df_check["seconds_since_landing"].values, df_check["value"].values, deg=1)
        lin_fn = np.poly1d(coeffs_lin)
        x_lin_fit = np.linspace(check_seconds, end_seconds, 100)
        y_lin_fit = lin_fn(x_lin_fit)
        timestamps_lin = plotly_timestamps([im2_landing + pd.Timedelta(seconds=float(sec)) for sec in x_lin_fit])
        fig.add_trace(go.Scatter(x=timestamps_lin, y=y_lin_fit, mode="lines", line=dict(color=color, dash="dot", width=2), name=f"{sensor} Linear Extrapolation"))

yaoki_surface_start = plotly_timestamp(YAOKI_FIRST_TELEMETRY_DATE)
yaoki_surface_stop = plotly_timestamp(YAOKI_LAST_TELEMETRY_DATE)
fig.add_vrect(x0=yaoki_surface_start, x1=yaoki_surface_stop, fillcolor="LightGray", opacity=0.3, layer="below", line_width=0)
fig.add_vline(x=plotly_timestamp(IM2_LANDING_DATE), line_width=2)
fig.add_vline(x=plotly_timestamp(IM2_THERMAL_SYSTEM_CHECK), line_width=2, line_dash="dot")
fig.update_layout(title="Lander Temperature Fits", xaxis_title="Remote Time (UTC)", yaxis_title="Temperature (deg C)", template="simple_white")
fig.update_layout(
    legend=dict(
        orientation="h",        # horizontal
        yanchor="top",
        y=-0.35,                 # below the x-axis
        xanchor="left",
        x=0
    )
)
fig.show()
